In [20]:
%pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.2 MB/s eta 0:00:00
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.9 MB/s eta 0:00:00
  Using cached pyvis-0.3.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 8.1 MB/s eta 0:00:00
  Using cached kiwisolver-1.4.9-cp312-cp312-manylinux2014_x

  Using cached parso-0.8.5-py2.py3-none-any.whl.metadata (8.3 kB)
  Using cached ptyprocess-0.7.0-py2.py3-none-any.whl.metadata (1.3 kB)
  Using cached executing-2.2.1-py2.py3-none-any.whl.metadata (8.9 kB)
  Using cached pure_eval-0.2.3-py3-none-any.whl.metadata (6.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 25.0 MB/s eta 0:00:00
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 55.3 MB/s eta 0:00:00m eta 0:00:010:01
Using cached pyvis-0.3.2-py3-none-any.whl (756 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 12.5 MB/s eta 0:00:00
Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (362 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 61.2 MB/s eta 0:00:00m eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.9/234.9 kB 32.5 M

In [33]:
import os
import json
from google import genai as genai
from dotenv import load_dotenv
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network

ModuleNotFoundError: No module named 'google'

In [8]:
prompt_path = "prompt-v2.txt"
json_path = "mortgage-q10-blade/knowledge-graph.json"
error_path = "mortgage-q10-blade/knowledge-graph.txt"
graph_path = "mortgage-q10-blade/knowledge-graph.html"

In [22]:
%matplotlib inline

In [23]:
def convert_to_json(text):
    try:
        graph_json = json.loads(text)
    except json.JSONDecodeError:
        with open(error_path, "w", encoding="utf-8") as f:
            f.write(text)
        return
    
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(graph_json, f, indent=2)
    
    print("graph successfully saved to knowledge-graph-v2.")
    return graph_json

In [25]:
%pip install -q -U google-genai

In [26]:
# DO NOT RUN UNLESS CALLING API

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    exit

genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-2.5-pro")

with open(prompt_path, "r", encoding="utf-8") as f:
    prompt = f.read()

print("Calling API:\n")
response = model.generate_content(prompt)
text = response.text
graph_json = convert_to_json(text)

NameError: name 'load_dotenv' is not defined

In [29]:
try:
    with open(json_path, 'r') as file:
        graph_json = json.load(file)
except FileNotFoundError:
    print(f"Error: The file '{json_path}' was not found.")
except json.JSONDecodeError:
    print(f"Error: Failed to decode JSON from the file '{json_path}'. Check for malformed JSON.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [30]:
# build graph -> NetworkX library
G = nx.DiGraph()

for node, edges in graph_json.items():
    for edge in edges:
        neighbor = edge.get("neighbor")
        relation = edge.get("edge", "")
        if neighbor:
            G.add_node(node)
            G.add_node(neighbor)
            G.add_edge(node, neighbor, label=relation)

# visualize graph
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=0.5, iterations=50)
edge_labels = nx.get_edge_attributes(G, "label")

nx.draw_networkx_nodes(G, pos, node_size=700, node_color="#66b3ff")
nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle="->", arrowsize=15)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

plt.title("Knowledge Graph")
plt.axis("off")
plt.tight_layout()
plt.show()


NameError: name 'nx' is not defined

In [37]:
net = Network(height="750px", width="100%", directed=True, bgcolor="#ffffff", font_color="#000000")
net.from_nx(G)

net.repulsion(
    node_distance=150,
    central_gravity=0.33,
    spring_length=200,
    spring_strength=0.05,
    damping=0.95
)

for src, dst, data in G.edges(data=True):
    label = data.get("label", "")
    if label:
        net.edges[-1]["title"] = label

net.show(graph_path, notebook=False)


mortgage-q10/knowledge-graph.html


In [38]:
# go through the generated graph and manually perform a cycle detection algorithm that prints out
# the cycles found

cycles = list(nx.simple_cycles(G))
if cycles:
    print("Cycles detected in the graph:")
    for cycle in cycles:
        print(" -> ".join(cycle))
else:
    print("No cycles detected in the graph.")

Cycles detected in the graph:
Cash Reserves -> Months of PITI in Reserve


In [39]:
# go through the generated graph and manually perform a cycle detection algorithm that prints out
# the cycles found

cycles = list(nx.simple_cycles(G))
if cycles:
    print("Cycles detected in the graph:")
    for cycle in cycles:
        print(" -> ".join(cycle))
else:
    print("No cycles detected in the graph.")

Cycles detected in the graph:
Cash Reserves -> Months of PITI in Reserve


In [40]:
# delete edges if they form cycles
for cycle in cycles:
    for i in range(len(cycle)):
        src = cycle[i]
        dst = cycle[(i + 1) % len(cycle)]
        if G.has_edge(src, dst):
            G.remove_edge(src, dst)
            print(f"Removed edge from {src} to {dst} to break the cycle.")
            break


Removed edge from Cash Reserves to Months of PITI in Reserve to break the cycle.


In [42]:
net.show(graph_path, notebook=False)

mortgage-q10/knowledge-graph.html
